# Step 6 — NammaMart ELT pipeline test

This notebook runs the ELT implementation locally. The raw layer is loaded from the CSV files as text; staging and mart transformations are executed from the checked-in SQL files.

Flow: load raw → transform staging with SQL → build marts with SQL → inspect row counts and business outputs.

In [1]:
from pathlib import Path
import sys
import duckdb

# Import the same ELT module used by the Airflow DAG.
sys.path.insert(0, str(Path('include').resolve()))
from nammamart_elt import run_elt

# Keep the ELT warehouse separate from the ETL warehouse for comparison.
warehouse = Path('assignment/warehouse_samy_elt_step6.duckdb')
metrics = run_elt(
    warehouse_path=warehouse,
    run_id='step6-local-test',
)
for name, value in metrics.items():
    print(f'{name}: {value:,}' if isinstance(value, int) else f'{name}: {value}')

warehouse_path: assignment/warehouse_samy_elt_step6.duckdb
raw_orders: 2,412
stg_orders: 2,400
passing_orders: 1,903
mart_store_rows: 384
mart_category_rows: 6
reconciliation_rows: 629


In [2]:
# Inspect every layer to prove that raw data, staging decisions and marts exist.
con = duckdb.connect(str(warehouse), read_only=True)
display(con.execute("""
    SELECT table_name, estimated_size
    FROM duckdb_tables()
    WHERE schema_name = 'main'
    ORDER BY table_name
""").fetchdf())

# Raw rows are retained while staging exposes pass/fail decisions.
display(con.execute("SELECT dq_status, count(*) AS rows FROM stg_orders GROUP BY dq_status").fetchdf())
display(con.execute("SELECT * FROM mart_category_margin ORDER BY category").fetchdf())
display(con.execute("SELECT * FROM mart_daily_store_kpis ORDER BY order_date, zone, store_id LIMIT 10").fetchdf())
display(con.execute("SELECT * FROM mart_top_customers LIMIT 10").fetchdf())
con.close()

,table_name,estimated_size
0,dq_results,37
1,mart_category_margin,6
2,mart_daily_store_kpis,384
3,mart_delivery_sla,13
4,mart_payment_reconciliation,629
5,mart_top_customers,10
6,pipeline_runs,1
7,raw_orders,2412
8,raw_payments,2228
9,raw_products,152


,dq_status,rows
0,PASS,1903
1,FAIL,497


,category,revenue,gross_margin
0,Dairy & Bakery,149028.75,39177.96
1,Fruits & Vegetables,75956.15,18226.14
2,Household,166866.55,42326.26
3,Personal Care,179965.60,39824.12
4,Snacks & Beverages,148381.10,33965.22
5,Staples,212686.00,54563.95


,order_date,zone,store_id,daily_net_revenue,delivered_orders
0,2026-09-01,East,S08,2984.70,4
1,2026-09-01,East,S09,2089.60,4
2,2026-09-01,East,S10,2141.60,3
3,2026-09-01,East,S11,2121.15,6
4,2026-09-01,North,S01,1410.80,3
5,2026-09-01,North,S02,2385.55,6
6,2026-09-01,North,S03,4583.80,6
7,2026-09-01,South,S05,803.00,2
8,2026-09-01,South,S06,2812.00,6
9,2026-09-01,West,S13,1469.10,3


,customer_id,full_name,email_masked,phone_masked,loyalty_tier,total_spend
0,C00310,Kiran Patil,ki***@example,91******35,Bronze,7258.95
1,C00072,Nandini Murthy,na***@example.com,98******05,Silver,6433.95
2,C00283,Nikhil Rao,ni***@example.com,91******43,Bronze,6241.40
3,C00236,Sneha Naidu,sn***@example.com,90******01,Platinum,6105.80
4,C00202,Priya Bhat,pr***@example.com,91******11,Silver,5988.75
5,C00328,Ritu Hegde,ri***@example.com,95******90,Bronze,5877.90
6,C00036,Deepa Desai,de***@example.com,90******55,Bronze,5835.70
7,C00115,Sanjay Kumar,sa***@example.com,96******85,Bronze,5783.70
8,C00105,Mahesh Gowda,ma***@example.com,97******56,Gold,5702.85
9,C00135,Deepa Desai,de***@example.com,90******72,Bronze,5693.20


In [3]:
# Re-running the initial ELT load replaces the derived tables, so this test
# confirms the initial pipeline does not duplicate rows on rerun.
second_metrics = run_elt(warehouse_path=warehouse, run_id='step6-local-rerun')
con = duckdb.connect(str(warehouse), read_only=True)
raw_count = con.execute('SELECT count(*) FROM raw_orders').fetchone()[0]
stg_count = con.execute('SELECT count(*) FROM stg_orders').fetchone()[0]
con.close()
print(f'Raw orders after rerun: {raw_count:,}')
print(f'Staging orders after rerun: {stg_count:,}')
print('The initial ELT run is deterministic; incremental append/merge is implemented in the next step.')

Raw orders after rerun: 2,412
Staging orders after rerun: 2,400
The initial ELT run is deterministic; incremental append/merge is implemented in the next step.
